# Step 1: train RealMLP and export it to numpy (`datathon-dl-env`)

The platform has no PyTorch, so RealMLP's prediction step was rewritten in plain numpy (`realmlp_numpy.py`). This notebook trains the 10 RealMLP seeds on all of `train.csv` (exactly as in `deep_learning_model.ipynb`), reads out every weight, and checks the numpy prediction against PyTorch on all 11,000 train + test rows.

Output: `realmlp_parts/seed0.npz` ... `seed9.npz` (weights) and `realmlp_parts/medians.npz` (the building × hour medians used to fill blanks). **Run with the `datathon-dl-env` interpreter.** About 30 seconds per seed; seeds already exported are reused unless `RETRAIN = True`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('.').resolve()))
import export_realmlp as ex
from realmlp_numpy import realmlp_inputs, realmlp_predict

SEEDS = list(range(10))
RETRAIN = False            # True: retrain every seed even if realmlp_parts/seed<N>.npz exists
OUT_DIR = Path('realmlp_parts')

In [2]:
from pytabkit import RealMLP_TD_Regressor

OUT_DIR.mkdir(exist_ok=True)
features = ex.RealMLPFeatures().fit(ex.train)
medians = features.as_numpy()
np.savez(OUT_DIR / 'medians.npz', **medians)
all_rows = pd.concat([ex.train.drop(columns='energy_usage'), ex.test_inputs], ignore_index=True)
x, codes = realmlp_inputs(all_rows, medians)

for seed in SEEDS:
    path = OUT_DIR / f'seed{seed}.npz'
    if path.exists() and not RETRAIN:
        weights = dict(np.load(path))
        saved = np.load(ex.ROOT / 'experiments' / f'realmlp_test_seed{seed}.npy')
        diff = np.abs(realmlp_predict(weights, x, codes)[len(ex.train):] - saved).max()
        print(f'seed {seed}: reused {path.name} | numpy vs saved PyTorch test predictions: largest difference {diff:.2e}')
        continue
    model = RealMLP_TD_Regressor(device='cpu', random_state=seed, n_threads=2, verbosity=0)
    model.fit(features.transform(ex.train), ex.train['energy_usage'].to_numpy(), cat_col_names=ex.CATEGORICAL_COLUMNS)
    torch_prediction = model.predict(features.transform(all_rows))
    weights = ex.extract_weights(model)
    np.savez(path, **weights)
    print(f'seed {seed}: trained | numpy vs PyTorch largest difference {np.abs(realmlp_predict(weights, x, codes) - torch_prediction).max():.2e}')

seed 0: reused seed0.npz | numpy vs saved PyTorch test predictions: largest difference 1.54e-05
seed 1: reused seed1.npz | numpy vs saved PyTorch test predictions: largest difference 1.27e-05
seed 2: reused seed2.npz | numpy vs saved PyTorch test predictions: largest difference 2.37e-05
seed 3: reused seed3.npz | numpy vs saved PyTorch test predictions: largest difference 1.76e-05
seed 4: reused seed4.npz | numpy vs saved PyTorch test predictions: largest difference 1.66e-05
seed 5: reused seed5.npz | numpy vs saved PyTorch test predictions: largest difference 1.60e-05
seed 6: reused seed6.npz | numpy vs saved PyTorch test predictions: largest difference 1.82e-05
seed 7: reused seed7.npz | numpy vs saved PyTorch test predictions: largest difference 1.43e-05
seed 8: reused seed8.npz | numpy vs saved PyTorch test predictions: largest difference 1.59e-05
seed 9: reused seed9.npz | numpy vs saved PyTorch test predictions: largest difference 1.44e-05
